In [1]:
!pip install -q kaggle
from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

!kaggle datasets download fernando2rad/brain-tumor-mri-images-44c
!mkdir /content/Datasets
!unzip brain-tumor-mri-images-44c.zip -d /content/Datasets
!rm brain-tumor-mri-images-44c.zip

In [ ]:
import torch
import math
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([transforms.Resize(256),
                                transforms.CenterCrop(224),
                                transforms.ToTensor(),
                                transforms.Normalize((0, 0, 0), (1, 1, 1))])

data_dir = '/content/Datasets/'
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Preprocessing All 3 datasets

In [2]:
#imports

import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from google.colab import userdata
import json

## Get datasets from Kaggle

In [3]:

# read from secrets — key never appears in the notebook
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key      = userdata.get('KAGGLE_KEY')

# create kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)

os.chmod('/root/.kaggle/kaggle.json', 600)
print("Done!")

Done!


In [ ]:
# install kaggle library
!pip install kaggle

# download the datasets
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!kaggle datasets download -d fernando2rad/brain-tumor-mri-images-44c

# unzip
!unzip brain-tumor-classification-mri.zip -d /content/dataset1
!unzip brain-tumor-mri-dataset.zip -d /content/dataset2
!unzip brain-tumor-mri-images-44c.zip -d /content/dataset3




Map between same-labels of datasets 1 and 2

In [5]:

print(os.listdir('/content/dataset1/Training'))
print(os.listdir('/content/dataset2/Training'))

['meningioma_tumor', 'no_tumor', 'glioma_tumor', 'pituitary_tumor']
['notumor', 'glioma', 'meningioma', 'pituitary']


In [6]:
#classes in dataset 1 and 2 are not in the same name add a mapping :
#standardize class names across datasets
CLASS_MAPPING = {
    # Dataset 1 names
    'glioma_tumor'      : 'glioma',
    'meningioma_tumor'  : 'meningioma',
    'pituitary_tumor'   : 'pituitary',
    'no_tumor'          : 'no_tumor',
    # Dataset 2 names
    'glioma'            : 'glioma',
    'meningioma'        : 'meningioma',
    'pituitary'         : 'pituitary',
    'notumor'           : 'no_tumor',
}

# standard label assignment — same across all datasets
STANDARD_CLASSES = {
    'glioma'     : 0,
    'meningioma' : 1,
    'pituitary'  : 2,
    'no_tumor'   : 3,
}

#Dataset 3 get's it's own label mapping from 0 to 43

## Preprocess images
when training any of our 3 models, always use the normalized versions


## Load Datasets & convert size & color

> Add blockquote



In [ ]:
#dataset 1
dataset1_path = '/content/dataset1/Training'

# class name mapping — dataset1 folder names → standard label numbers
dataset1_class_to_label = {
    'glioma_tumor'     : 0,
    'meningioma_tumor' : 1,
    'pituitary_tumor'  : 2,
    'no_tumor'         : 3,
}

dataset1_images = []   # will hold numpy arrays
dataset1_labels = []   # will hold label numbers

for folder_name, label in dataset1_class_to_label.items():
    folder_path = os.path.join(dataset1_path, folder_name)
    image_files = [f for f in os.listdir(folder_path)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"Dataset 1 — {folder_name}: {len(image_files)} images")

    for img_name in tqdm(image_files, desc=folder_name):
        img_path = os.path.join(folder_path, img_name)
        # convert to grayscale
        img      = Image.open(img_path).convert('L')
        # resize to 224x224
        img      = img.resize((224, 224), Image.BILINEAR)
        # create numpy array
        img_array = np.array(img, dtype=np.float32)
        dataset1_images.append(img_array)
        dataset1_labels.append(label)
#We end up with a numpy array of own images.
print(f"\nDataset 1 total: {len(dataset1_images)} images")

In [ ]:
#load dataset 2
dataset2_path = '/content/dataset2/Training'

# same label numbers as dataset 1 — just different folder names
dataset2_class_to_label = {
    'glioma'     : 0,   # same as glioma_tumor above
    'meningioma' : 1,   # same as meningioma_tumor above
    'pituitary'  : 2,   # same as pituitary_tumor above
    'notumor'    : 3,   # same as no_tumor above
}

dataset2_images = []
dataset2_labels = []

for folder_name, label in dataset2_class_to_label.items():
    folder_path = os.path.join(dataset2_path, folder_name)
    image_files = [f for f in os.listdir(folder_path)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"Dataset 2 — {folder_name}: {len(image_files)} images")

    for img_name in tqdm(image_files, desc=folder_name):
        img_path  = os.path.join(folder_path, img_name)
        img       = Image.open(img_path).convert('L')
        img       = img.resize((224, 224), Image.BILINEAR)
        img_array = np.array(img, dtype=np.float32)
        dataset2_images.append(img_array)
        dataset2_labels.append(label)

print(f"\nDataset 2 total: {len(dataset2_images)} images")

In [ ]:
#load dataset 3

dataset3_path = '/content/dataset3'

# 44 classes — just number them 0 to 43 alphabetically
dataset3_folders = sorted([
    d for d in os.listdir(dataset3_path)
    if os.path.isdir(os.path.join(dataset3_path, d))
])
dataset3_class_to_label = {folder: idx for idx, folder in enumerate(dataset3_folders)}

print("Dataset 3 classes:")
for folder, label in dataset3_class_to_label.items():
    print(f"  {label} = {folder}")

dataset3_images = []
dataset3_labels = []

for folder_name, label in dataset3_class_to_label.items():
    folder_path = os.path.join(dataset3_path, folder_name)
    image_files = [f for f in os.listdir(folder_path)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for img_name in tqdm(image_files, desc=folder_name):
        img_path  = os.path.join(folder_path, img_name)
        img       = Image.open(img_path).convert('L')
        img       = img.resize((224, 224), Image.BILINEAR)
        img_array = np.array(img, dtype=np.float32)
        dataset3_images.append(img_array)
        dataset3_labels.append(label)

print(f"\nDataset 3 total: {len(dataset3_images)} images")

## Normalize Datasets

### Normalize Dataset1






In [ ]:


from sklearn.model_selection import train_test_split
import numpy as np
import torch

# Converting to numpy arrays
images = np.array(dataset1_images)
labels = np.array(dataset1_labels)

# First stratified 70-30 split
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

# Second stratifiet split
val_ratio = 0.10 / 0.30

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=1 - val_ratio,
    stratify=y_temp,
    random_state=42
)

print("Dataset 1 → Train:", len(X_train), "Test:", len(X_test), "Val:", len(X_val))

# Normalizing each split
train_norm = X_train / 255.0
test_norm   = X_test   / 255.0
val_norm  = X_val  / 255.0

# Converting to tensors
train_tensors = torch.tensor(train_norm).unsqueeze(1)
test_tensors   = torch.tensor(test_norm).unsqueeze(1)
val_tensors  = torch.tensor(val_norm).unsqueeze(1)

train_labels_tensor = torch.tensor(y_train, dtype=torch.long)
test_labels_tensor   = torch.tensor(y_test, dtype=torch.long)
val_labels_tensor  = torch.tensor(y_val, dtype=torch.long)

# Saving the splits
torch.save({
    'images': train_tensors,
    'labels': train_labels_tensor,
    'class_to_label': dataset1_class_to_label,
}, '/content/dataset1_train.pt')

torch.save({
    'images': test_tensors,
    'labels': test_labels_tensor,
    'class_to_label': dataset1_class_to_label,
}, '/content/dataset1_test.pt')

torch.save({
    'images': val_tensors,
    'labels': val_labels_tensor,
    'class_to_label': dataset1_class_to_label,
}, '/content/dataset1_val.pt')

print("Saved dataset1_train.pt, dataset1_test.pt, dataset1_val.pt")


Dataset 1 → Train: 2009 Test: 574 Val: 287
Saved dataset1_train.pt, dataset1_test.pt, dataset1_val.pt


### Normalize Dataset2


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import torch

# Converting to numpy arrays
images = np.array(dataset2_images)
labels = np.array(dataset2_labels)

# First stratified 70-30 split
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

# Second stratified split
val_ratio = 0.10 / 0.30

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=1 - val_ratio,
    stratify=y_temp,
    random_state=42
)

print("Dataset 2 → Train:", len(X_train), "Test:", len(X_test), "Val:", len(X_val))

# Normalizing each split
train_norm = X_train / 255.0
val_norm   = X_val   / 255.0
test_norm  = X_test  / 255.0

# Converting to tensors
train_tensors = torch.tensor(train_norm).unsqueeze(1)
test_tensors  = torch.tensor(test_norm).unsqueeze(1)
val_tensors   = torch.tensor(val_norm).unsqueeze(1)

train_labels_tensor = torch.tensor(y_train, dtype=torch.long)
test_labels_tensor  = torch.tensor(y_test, dtype=torch.long)
val_labels_tensor   = torch.tensor(y_val, dtype=torch.long)

# Saving the splits
torch.save({
    'images': train_tensors,
    'labels': train_labels_tensor,
    'class_to_label': dataset2_class_to_label,
}, '/content/dataset2_train.pt')

torch.save({
    'images': test_tensors,
    'labels': test_labels_tensor,
    'class_to_label': dataset2_class_to_label,
}, '/content/dataset2_test.pt')

torch.save({
    'images': val_tensors,
    'labels': val_labels_tensor,
    'class_to_label': dataset2_class_to_label,
}, '/content/dataset2_val.pt')

print("Saved dataset2_train.pt, dataset2_test.pt, dataset2_val.pt")


Dataset 2 → Train: 3920 Test: 1120 Val: 560
Saved dataset2_train.pt, dataset2_test.pt, dataset2_val.pt


### Normalise dataset3

In [ ]:

from sklearn.model_selection import train_test_split
import numpy as np
import torch

# Converting to numpy arrays
images = np.array(dataset3_images)
labels = np.array(dataset3_labels)

# First stratified 70-30 split
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.3, stratify=labels, random_state=42
)

# Second stratified split
val_ratio = 0.1 / 0.3

X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test, test_size=1 - val_ratio, stratify=y_test, random_state=42
)

print ("Train:", len(X_train), "Test:", len(X_test), "Val:", len(X_val))

# Normalizing each split
train_norm = X_train / 255.0
test_norm = X_test / 255.0
val_norm = X_val / 255.0

# Converting to tensors
train_tensors = torch.tensor(train_norm).unsqueeze(1)
test_tensors = torch.tensor(test_norm).unsqueeze(1)
val_tensors = torch.tensor(val_norm).unsqueeze(1)

train_labels_tensor = torch.tensor(y_train, dtype=torch.long)
test_labels_tensor = torch.tensor(y_test, dtype=torch.long)
val_labels_tensor = torch.tensor(y_val, dtype=torch.long)

# Saving the splits
torch.save({
    'images'         : train_tensors,
    'labels'         : train_labels_tensor,
    'class_to_label' : dataset3_class_to_label,
}, '/content/dataset3_train.pt')

torch.save({
    'images': test_tensors,
    'labels': test_labels_tensor,
    'class_to_label': dataset3_class_to_label,
    }, '/content/dataset3_test.pt')

torch.save({
    'images': val_tensors,
    'labels': val_labels_tensor,
    'class_to_label': dataset3_class_to_label,
    }, '/content/dataset3_val.pt')

print("Saved dataset3_train.pt, dataset3_test.pt, dataset3_val.pt")

Train: 3134 Test: 896 Val: 448
Saved dataset3_train.pt, dataset3_test.pt, dataset3_val.pt


In [ ]:
#Visualise the results
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle("Sample images after preprocessing", fontsize=14)

for row, (tensors, name) in enumerate([
    (dataset1_tensors, 'Dataset 1'),
    (dataset2_tensors, 'Dataset 2'),
    (dataset3_tensors, 'Dataset 3'),
]):
    indices = torch.randperm(len(tensors))[:4]
    for col, idx in enumerate(indices):
        img = tensors[idx].squeeze(0).numpy()
        axes[row][col].imshow(img, cmap='gray')
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(
                f"{name}\nmean={img.mean():.2f}",
                fontsize=8, loc='left'
            )

plt.tight_layout()
plt.savefig('/content/verification.png', dpi=150, bbox_inches='tight')
plt.show()

# PreProcess Testing data

#### PreProcess Dataset 1

In [ ]:


# --- TESTING SET for dataset 1 ---
dataset1_test_path = '/content/dataset1/Testing'

dataset1_test_images = []
dataset1_test_labels = []

for folder_name, label in dataset1_class_to_label.items():
    folder_path = os.path.join(dataset1_test_path, folder_name)
    image_files = [f for f in os.listdir(folder_path)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"Dataset 1 TEST — {folder_name}: {len(image_files)} images")

    for img_name in tqdm(image_files, desc=folder_name):
        img_path = os.path.join(folder_path, img_name)
        img = Image.open(img_path).convert('L')
        img = img.resize((224, 224), Image.BILINEAR)
        img_array = np.array(img, dtype=np.float32)
        dataset1_test_images.append(img_array)
        dataset1_test_labels.append(label)

# normalize 0-1
dataset1_test_normalized = np.array(dataset1_test_images) / 255.0

dataset1_test_tensors = torch.tensor(dataset1_test_normalized).unsqueeze(1)
dataset1_test_labels_tensor = torch.tensor(dataset1_test_labels, dtype=torch.long)

torch.save({
    'images'         : dataset1_test_tensors,
    'labels'         : dataset1_test_labels_tensor,
    'class_to_label' : dataset1_class_to_label,
}, '/content/dataset1_test_preprocessed.pt')

print("Saved dataset1_test_preprocessed.pt")


Dataset 1 TEST — glioma_tumor: 100 images


glioma_tumor: 100%|██████████| 100/100 [00:00<00:00, 236.44it/s]


Dataset 1 TEST — meningioma_tumor: 115 images


meningioma_tumor: 100%|██████████| 115/115 [00:00<00:00, 330.40it/s]


Dataset 1 TEST — pituitary_tumor: 74 images


pituitary_tumor: 100%|██████████| 74/74 [00:00<00:00, 103.35it/s]


Dataset 1 TEST — no_tumor: 105 images


no_tumor: 100%|██████████| 105/105 [00:00<00:00, 383.71it/s]


Saved dataset1_test_preprocessed.pt


PreProcess Dataset 2

In [ ]:
# --- TESTING SET for dataset 2 ---
dataset2_test_path = '/content/dataset2/Testing'

dataset2_test_images = []
dataset2_test_labels = []

for folder_name, label in dataset2_class_to_label.items():
    folder_path = os.path.join(dataset2_test_path, folder_name)
    image_files = [f for f in os.listdir(folder_path)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"Dataset 2 TEST — {folder_name}: {len(image_files)} images")

    for img_name in tqdm(image_files, desc=folder_name):
        img_path = os.path.join(folder_path, img_name)
        img = Image.open(img_path).convert('L')
        img = img.resize((224, 224), Image.BILINEAR)
        img_array = np.array(img, dtype=np.float32)
        dataset2_test_images.append(img_array)
        dataset2_test_labels.append(label)

# normalize 0–1
dataset2_test_normalized = np.array(dataset2_test_images) / 255.0

dataset2_test_tensors = torch.tensor(dataset2_test_normalized).unsqueeze(1)
dataset2_test_labels_tensor = torch.tensor(dataset2_test_labels, dtype=torch.long)

torch.save({
    'images'         : dataset2_test_tensors,
    'labels'         : dataset2_test_labels_tensor,
    'class_to_label' : dataset2_class_to_label,
}, '/content/dataset2_test_preprocessed.pt')

print("Saved dataset2_test_preprocessed.pt")


Dataset 2 TEST — glioma: 400 images


glioma: 100%|██████████| 400/400 [00:02<00:00, 193.91it/s]


Dataset 2 TEST — meningioma: 400 images


meningioma: 100%|██████████| 400/400 [00:01<00:00, 205.66it/s]


Dataset 2 TEST — pituitary: 400 images


pituitary: 100%|██████████| 400/400 [00:01<00:00, 265.25it/s]


Dataset 2 TEST — notumor: 400 images


notumor: 100%|██████████| 400/400 [00:01<00:00, 398.45it/s]


Saved dataset2_test_preprocessed.pt


PreProcess Dataset 3

Save to google drive

In [ ]:
from google.colab import drive

# Mount Google Drive, forcing a remount to ensure a clean connection.
drive.mount('/content/drive', force_remount=True)

import shutil
import os

save_dir = '/content/drive/MyDrive/brain_mri_preprocessed'
os.makedirs(save_dir, exist_ok=True)

for filename in [
    'dataset1_preprocessed.pt',
    'dataset2_preprocessed.pt',
    'dataset3_preprocessed.pt',
    'dataset1_test_preprocessed.pt',
    'normalization_stats.pt',
    'verification.png',
]:
    src  = f'/content/{filename}'
    dest = f'{save_dir}/{filename}'
    shutil.copy2(src, dest)
    print(f"Saved: {dest}")

print("\nAll done!")

Mounted at /content/drive
Saved: /content/drive/MyDrive/brain_mri_preprocessed/dataset1_preprocessed.pt
Saved: /content/drive/MyDrive/brain_mri_preprocessed/dataset2_preprocessed.pt


FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset3_preprocessed.pt'

## Data Augmentation

Setup MONAI imports

In [ ]:
!python -c "import monai" || pip install -q "monai-weekly[ignite, tqdm]"

import logging
import numpy as np
import os
from pathlib import Path
import sys
import tempfile
import torch

from monai.apps import MedNISTDataset
from monai.config import print_config
from monai.engines import SupervisedTrainer
from monai.handlers import StatsHandler
from monai.inferers import SimpleInferer
from monai.networks import eval_mode
from monai.networks.nets import densenet121
from monai.transforms import (
    Compose,
    LoadImage,
    EnsureChannelFirst,
    Resize,
    NormalizeIntensity,
    RandFlipd,
    RandRotate,
    RandZoom,
    RandGaussianNoise,
    RandAdjustContrast,
    RandShiftIntensity,
    EnsureType,
)
from monai.data import ImageDataset


print_config()

### Dataset 1 Data augmentation

In [ ]:
from torch.utils.data import DataLoader

# --- Step 1: Load preprocessed tensors ---
checkpoint = torch.load('/content/drive/MyDrive/brain_mri_preprocessed/dataset1_preprocessed.pt')
train_images = checkpoint['images']
train_labels = checkpoint['labels']
class_to_label = checkpoint['class_to_label']

test_checkpoint = torch.load('/content/drive/MyDrive/brain_mri_preprocessed/dataset1_test_preprocessed.pt')
test_images = test_checkpoint['images']
test_labels = test_checkpoint['labels']

print(f"Train shape : {train_images.shape}")
print(f"Test shape  : {test_images.shape}")
print(f"Classes     : {class_to_label}")

# --- Step 2: Augmentation applied on-the-fly via custom Dataset ---
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

train_transforms = Compose([
    RandRotate(range_x=0.15, prob=0.5, keep_size=True), #simulates slight head tilt during scanning, realistic
    RandZoom(min_zoom=0.95, max_zoom=1.05, prob=0.3, keep_size=True), # simulates minor field-of-view differences between scanners,
    RandGaussianNoise(mean=0.0, std=0.01, prob=0.2), #simulates scanner noise, realistic
    RandAdjustContrast(gamma=(0.9, 1.1), prob=0.2), # simulates scanner/protocol differences, realistic
    RandShiftIntensity(offsets=0.05, prob=0.2), # simulates brightness variation between scanners, realistic
])

# augmentation only for training, none for test
train_dataset = AugmentedDataset(train_images, train_labels, transform=train_transforms);
test_dataset  = AugmentedDataset(test_images,  test_labels,  transform=None);

# --- Step 3: DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True);
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False);

print(f"Train: {len(train_dataset)} images")
print(f"Test:  {len(test_dataset)} images")

## Data Augmentation for class imbalance
Apply heavier augmentation to the minority class (No tumor) OR have the model pickup from this particular clas smore often..


Trianing percentages for dataset 1 and 2

In [ ]:
import os

for name, path in [('Dataset 1', '/content/dataset1'), ('Dataset 2', '/content/dataset2')]:
    train_count = sum(
        len([f for f in os.listdir(os.path.join(path, 'Training', c))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for c in os.listdir(os.path.join(path, 'Training'))
        if os.path.isdir(os.path.join(path, 'Training', c))
    )
    test_path = os.path.join(path, 'Testing')
    if os.path.exists(test_path):
        test_count = sum(
            len([f for f in os.listdir(os.path.join(test_path, c))
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            for c in os.listdir(test_path)
            if os.path.isdir(os.path.join(test_path, c))
        )
    else:
        test_count = 0

    total = train_count + test_count
    print(f"{name}: {train_count} train ({train_count/total*100:.1f}%) | "
          f"{test_count} test ({test_count/total*100:.1f}%) | "
          f"{total} total")
